# 03 Temporal Dataset and LSTM Architecture

Prepare chronological PM2.5 sequences and inspect the initial LSTM architecture. This notebook does not train or evaluate a model.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Could not find the project root. Open Jupyter from the project folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_DIR,
    SEQUENCE_LENGTH,
    TARGET_COLUMN,
    TEMPORAL_FEATURE_COLUMNS,
)
from src.data_loader import load_all_station_data
from src.lstm_model import build_lstm_model
from src.preprocessing import build_datetime_column
from src.temporal_data import prepare_temporal_data

# Start from raw station data so Stage 5 can fit imputers and scalers on training dates only.
df = build_datetime_column(load_all_station_data(RAW_DATA_DIR))
df = df.sort_values(['station', 'datetime']).drop_duplicates(['station', 'datetime']).reset_index(drop=True)
print(f'Dataset shape: {df.shape}')
print(f'Stations: {df["station"].nunique()}')
print(f'Date range: {df["datetime"].min()} to {df["datetime"].max()}')


Dataset shape: (420768, 19)
Stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 23:00:00


## Create chronological train, validation, and test sequences

The split is by timestamp, not random. Missing input features are filled with medians from the training period; feature and target scalers are also fitted only on training rows. Rows with a missing target are not used as labels. Each station is sequenced separately, windows with missing hourly steps are skipped, and validation/test windows use only observations inside their own split.


In [2]:
temporal_data = prepare_temporal_data(
    df,
    feature_columns=TEMPORAL_FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    sequence_length=SEQUENCE_LENGTH,
)

for split_name in ('train', 'validation', 'test'):
    X = temporal_data[f'X_{split_name}']
    y = temporal_data[f'y_{split_name}']
    print(f'{split_name}: X={X.shape}, y={y.shape}')

print('Split boundaries:', temporal_data['split_timestamps'])
print('Input features:', TEMPORAL_FEATURE_COLUMNS)
print('Forecast target:', TARGET_COLUMN)


train: X=(288081, 24, 11), y=(288081,)
validation: X=(61538, 24, 11), y=(61538,)
test: X=(61585, 24, 11), y=(61585,)
Split boundaries: {'train_end': Timestamp('2015-12-18 15:00:00'), 'validation_start': Timestamp('2015-12-18 16:00:00'), 'validation_end': Timestamp('2016-07-24 19:00:00'), 'test_start': Timestamp('2016-07-24 20:00:00')}
Input features: ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
Forecast target: PM2.5


## Build the initial LSTM

The model uses a 24-hour input window and predicts scaled PM2.5 for the next hour. It is not compiled or trained in this stage; training belongs to Stage 6.


In [3]:
model = build_lstm_model(
    sequence_length=SEQUENCE_LENGTH,
    n_features=len(TEMPORAL_FEATURE_COLUMNS),
)
model.summary()


Model: "pm25_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,961 (128.75 KB)

 Trainable params: 32,961 (128.75 KB)

 Non-trainable params: 0 (0.00 B)